Homework 2 - Car Fuel Efficiency

In [1]:
import pandas as pd
import numpy as np

# 0. CARGA Y PREPARACIÓN DEL DATASET
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[cols]

# QUESTION 1
missing_col = df.isnull().sum()[df.isnull().sum() > 0].index.tolist()[0]
print(f"Q1 - Columna con missing values: '{missing_col}'")

# QUESTION 2
median_hp = df['horsepower'].median()
print(f"Q2 - Mediana de horsepower: {median_hp}")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Q1 - Columna con missing values: 'horsepower'
Q2 - Mediana de horsepower: 254.0


In [2]:
# ==========================================
# FUNCIONES AUXILIARES (Regresión Lineal)
# ==========================================
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    reg[0, 0] = 0  # No regularizar el bias
    XTX = XTX + reg
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def predict(X, w0, w):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    return X.dot(np.concatenate([[w0], w]))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def get_splits(seed=42):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df.iloc[idx[:n_train]].copy()
    df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
    df_test = df.iloc[idx[n_train + n_val:]].copy()
    return df_train, df_val, df_test

target_col = 'horsepower'

In [3]:
# ==========================================
# QUESTION 3: Imputación (0 vs Media)
# ==========================================
df_train, df_val, df_test = get_splits(seed=42)

# Opción A: Rellenar con 0
df_tr_0, df_va_0 = df_train.copy(), df_val.copy()
df_tr_0[target_col] = df_tr_0[target_col].fillna(0)
df_va_0[target_col] = df_va_0[target_col].fillna(0)

X_tr_0 = df_tr_0.drop(columns=['fuel_efficiency_mpg']).values
y_tr_0 = df_tr_0['fuel_efficiency_mpg'].values
X_va_0 = df_va_0.drop(columns=['fuel_efficiency_mpg']).values
y_va_0 = df_va_0['fuel_efficiency_mpg'].values

w0_0, w_0 = train_linear_regression(X_tr_0, y_tr_0)
rmse_0 = round(rmse(y_va_0, predict(X_va_0, w0_0, w_0)), 3)

# Opción B: Rellenar con la media
mean_val = df_train[target_col].mean()
df_tr_m, df_va_m = df_train.copy(), df_val.copy()
df_tr_m[target_col] = df_tr_m[target_col].fillna(mean_val)
df_va_m[target_col] = df_va_m[target_col].fillna(mean_val)

X_tr_m = df_tr_m.drop(columns=['fuel_efficiency_mpg']).values
y_tr_m = df_tr_m['fuel_efficiency_mpg'].values
X_va_m = df_va_m.drop(columns=['fuel_efficiency_mpg']).values
y_va_m = df_va_m['fuel_efficiency_mpg'].values

w0_m, w_m = train_linear_regression(X_tr_m, y_tr_m)
rmse_m = round(rmse(y_va_m, predict(X_va_m, w0_m, w_m)), 3)

print(f"Q3 - RMSE con 0: {rmse_0} | RMSE con media: {rmse_m}")
print(f"-> Mejor opción: {'With mean' if rmse_m < rmse_0 else 'With 0'}")

Q3 - RMSE con 0: 2.205 | RMSE con media: 2.202
-> Mejor opción: With mean


In [4]:
# ==========================================
# QUESTION 4: Regularización (diferentes 'r')
# ==========================================
df_train, df_val, df_test = get_splits(seed=42)
df_train[target_col] = df_train[target_col].fillna(0)
df_val[target_col] = df_val[target_col].fillna(0)

X_tr = df_train.drop(columns=['fuel_efficiency_mpg']).values
y_tr = df_train['fuel_efficiency_mpg'].values
X_va = df_val.drop(columns=['fuel_efficiency_mpg']).values
y_va = df_val['fuel_efficiency_mpg'].values

r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
rmse_r = {}

for r in r_values:
    w0, w = train_linear_regression_reg(X_tr, y_tr, r=r)
    rmse_r[r] = round(rmse(y_va, predict(X_va, w0, w)), 4)

best_r = min(rmse_r, key=rmse_r.get)
print(f"Q4 - RMSE por r: {rmse_r}")
print(f"-> Mejor r: {best_r}")

Q4 - RMSE por r: {0: np.float64(2.2053), 0.01: np.float64(2.2053), 0.1: np.float64(2.2053), 1: np.float64(2.2053), 5: np.float64(2.2053), 10: np.float64(2.2053), 100: np.float64(2.2053)}
-> Mejor r: 0


In [5]:
# ==========================================
# QUESTION 5: Estabilidad del modelo (seeds)
# ==========================================
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmse_scores = []

for seed in seeds:
    df_tr, df_va, _ = get_splits(seed=seed)
    df_tr[target_col] = df_tr[target_col].fillna(0)
    df_va[target_col] = df_va[target_col].fillna(0)
    
    X_tr = df_tr.drop(columns=['fuel_efficiency_mpg']).values
    y_tr = df_tr['fuel_efficiency_mpg'].values
    X_va = df_va.drop(columns=['fuel_efficiency_mpg']).values
    y_va = df_va['fuel_efficiency_mpg'].values
    
    w0, w = train_linear_regression(X_tr, y_tr)
    rmse_scores.append(rmse(y_va, predict(X_va, w0, w)))

std_rmse = round(np.std(rmse_scores), 3)
print(f"Q5 - Desviación estándar de los RMSE: {std_rmse}")

Q5 - Desviación estándar de los RMSE: 0.029


In [6]:
# ==========================================
# QUESTION 6: RMSE en Test (seed 9, r=0.001)
# ==========================================
df_train, df_val, df_test = get_splits(seed=9)

df_train_val = pd.concat([df_train, df_val])
df_train_val[target_col] = df_train_val[target_col].fillna(0)
df_test[target_col] = df_test[target_col].fillna(0)

X_tr_val = df_train_val.drop(columns=['fuel_efficiency_mpg']).values
y_tr_val = df_train_val['fuel_efficiency_mpg'].values
X_te = df_test.drop(columns=['fuel_efficiency_mpg']).values
y_te = df_test['fuel_efficiency_mpg'].values

w0_6, w_6 = train_linear_regression_reg(X_tr_val, y_tr_val, r=0.001)
rmse_test = round(rmse(y_te, predict(X_te, w0_6, w_6)), 3)

print(f"Q6 - RMSE en el dataset de Test: {rmse_test}")

Q6 - RMSE en el dataset de Test: 2.236
